In [161]:
import glob
import pandas as pd

# Load data

In [162]:
def load_evidence(fn):
    ds_name = fn.split("/")[-3]
    df = pd.read_json(fn)
    # Unwrap the 'source' column (contains a dictionary) into separate columns
    source_df = df['source'].apply(pd.Series)
    extracted_df = df['extracted'].apply(pd.Series).add_prefix('extracted_')
    # derived_df = df['derived'].apply(pd.Series).add_prefix('derived_')
    # Combine the original DataFrame with the unwrapped columns
    # df = pd.concat([df.drop(columns=['source', 'extracted', 'derived']), source_df, extracted_df, derived_df], axis=1)
    df = pd.concat([df.drop(columns=['source', 'extracted']), source_df, extracted_df], axis=1)
    df["ds"] = ds_name
    del df["derived"]
    return df

In [163]:
fns = glob.glob("../../data/*/evidence_human/evidence.json")


In [164]:
dfs = []
for fn in fns:
    dfs.append(load_evidence(fn))
df = pd.concat(dfs)

# Perform token alignment

In [165]:
tdf = df.query("source_type == 'text'").copy()

In [166]:
tdf.head(2)

,source_type,source_rationale,source_id,extracted_organism,extracted_cell_type_label,extracted_cell_source,extracted_cell_state,extracted_feature_name,extracted_feature_type,ds
0,text,Monocyte 1 strongly expressed FCGR3A (CD16) an...,text,homo_sapiens,Monocyte 1,kidney,none,FCGR3A,gene,kidney_Wu2018
1,text,"Interestingly, ABCA1, which mediates sterol ef...",text,homo_sapiens,Monocyte 1,kidney,none,ABCA1,gene,kidney_Wu2018


In [167]:
from extract.extract import align, norm_text

In [168]:
tdf["source_cell_type_label"] = tdf.apply(
    lambda x: align(norm_text(x["source_rationale"]), norm_text(x["extracted_cell_type_label"]))[0], 
    axis=1)
tdf["source_feature_name"]    = tdf.apply(
    lambda x: align(norm_text(x["source_rationale"]), norm_text(x["extracted_feature_name"]))[0], 
    axis=1)

In [169]:
tdf["found_feature_name"] = tdf["source_feature_name"] == tdf["extracted_feature_name"]
tdf["found_cell_type_label"] = tdf["source_cell_type_label"] == tdf["extracted_cell_type_label"]

In [170]:
temp = tdf[tdf["source_cell_type_label"] != tdf["extracted_cell_type_label"]]
temp = temp[temp["ds"] == "adipose_Merrick2019"]
temp[temp["source_type"] == "text"]

,source_type,source_rationale,source_id,extracted_organism,extracted_cell_type_label,extracted_cell_source,extracted_cell_state,extracted_feature_name,extracted_feature_type,ds,source_cell_type_label,source_feature_name,found_feature_name,found_cell_type_label


In [171]:
agg = tdf.groupby("ds").agg({
    "found_feature_name": ["count", "sum"], 
    "found_cell_type_label": ["count", "sum"], 
    }).rename(columns={"count": "total", "sum": "found"}, level=1)
agg[("found_feature_name", "frac")] = agg[("found_feature_name", "found")] / agg[("found_feature_name", "total")]
agg[("found_cell_type_label", "frac")] = agg[("found_cell_type_label", "found")] / agg[("found_cell_type_label", "total")]
agg[("found_feature_name", "complete")] = agg[("found_feature_name", "frac")] == 1
agg[("found_cell_type_label", "complete")] = agg[("found_cell_type_label", "frac")] == 1
agg["all_complete"] = agg[("found_feature_name", "complete")] & agg[("found_cell_type_label", "complete")]
agg.sort_index(axis=1)


all_complete found_cell_type_label                  \
                                                   complete found      frac   
ds                                                                            
adipose_Emont2022                True                  True    17  1.000000   
adipose_Hildreth2021             True                  True    72  1.000000   
adipose_Jaitin2019               True                  True    49  1.000000   
adipose_Massier2023              True                  True    29  1.000000   
adipose_Merrick2019             False                  True    29  1.000000   
adipose_Vijay2019               False                 False    74  0.936709   
bladder_Yu2019                  False                 False    24  0.585366   
bone_He2021                     False                 False     0  0.000000   
cortex_Booeshaghi2021           False                 False     0  0.000000   
eye_Gautam2021                  False                 False     4  0.160000   
heart_Tucker2020                False                 False     4  0.054054   
kidney_Wu2018                   False                 False     2  0.064516   
liver_Guillams2022              False                 False     0  0.000000   
lung_Adams2020                  False                 False     0  0.000000   
ovary_Wagner2020                False                 False     0  0.000000   
pancreas_Segerstolpe2016        False                 False     0  0.000000   
placenta_Liu2018                False                 False     0  0.000000   
testis_Shamis2020               False                 False    60  0.517241   
yolksac_Goh2023                 False                 False    92  0.958333   

                               found_feature_name                        
                         total           complete found      frac total  
ds                                                                       
adipose_Emont2022           17               True    17  1.000000    17  
adipose_Hildreth2021        72               True    72  1.000000    72  
adipose_Jaitin2019          49               True    49  1.000000    49  
adipose_Massier2023         29               True    29  1.000000    29  
adipose_Merrick2019         29              False    26  0.896552    29  
adipose_Vijay2019           79               True    79  1.000000    79  
bladder_Yu2019              41              False    24  0.585366    41  
bone_He2021                 71              False     3  0.042254    71  
cortex_Booeshaghi2021        4              False     0  0.000000     4  
eye_Gautam2021              25              False     2  0.080000    25  
heart_Tucker2020            74              False    18  0.243243    74  
kidney_Wu2018               31              False     9  0.290323    31  
liver_Guillams2022           5              False     0  0.000000     5  
lung_Adams2020              37              False     5  0.135135    37  
ovary_Wagner2020            65              False    16  0.246154    65  
pancreas_Segerstolpe2016   124              False     8  0.064516   124  
placenta_Liu2018           102              False    12  0.117647   102  
testis_Shamis2020          116              False    69  0.594828   116  
yolksac_Goh2023             96              False    94  0.979167    96

In [172]:
nobs = tdf.shape[0]
found_ct = tdf["found_cell_type_label"].sum()
found_fn = tdf["found_feature_name"].sum()
frac_ct = found_ct / nobs * 100
frac_fn = found_fn / nobs * 100


In [173]:
print(f"Features   found: {found_fn}/{nobs} ({frac_fn:.1f}%)")
print(f"Cell types found: {found_ct}/{nobs} ({frac_ct:.1f}%)")

Features   found: 532/1066 (49.9%)
Cell types found: 456/1066 (42.8%)
